# Experimento: FT-Transformer, fase 2 (regularização)

**O que a fase 1 mostrou.** As 10 versões empataram com o trial 0 no placar (KS entre 0,537 e 0,544). O sinal de sobreajuste está na **época de parada**: o trial 0 para na época mediana 6, e algumas dobras param já na época 1. A rede aprende o que consegue em poucas épocas, e logo a perda na separação interna começa a piorar. Como a parada corta o treino antes, o sobreajuste quase não aparece no MSE (0,156 no treino e 0,165 na validação).

**Hipótese.** Com regularização, a rede consegue treinar por mais tempo sem decorar e talvez aprenda mais.

## Perguntas

1. **Dá para ver o sobreajuste?** Curvas da entropia cruzada no treino e na separação de parada, época a época, sem parar antes: o trial 0 contra as versões mais regularizadas.
2. **Algum regularizador ganha do trial 0 no placar?** Cada um sozinho, em duas intensidades, e depois duas combinações:

| Regularizador | O que faz | Intensidades |
|---|---|---|
| Dropout (atenção, FFN, residual) | desliga ao acaso parte das conexões em cada passo | médio 0,3 / 0,3 / 0,1; pesado 0,5 / 0,5 / 0,2 (artigo: 0,2 / 0,1 / 0) |
| L2 na perda | soma λ·Σw² à perda: puxa todos os pesos para perto de zero | 1e-4 e 1e-3 |
| L1 na perda | soma λ·Σ\|w\| à perda: zera os pesos pouco úteis | 1e-6 e 1e-5 |
| Weight decay do AdamW | encolhe os pesos lr·wd por passo, por fora do gradiente | 1,0 (o 1e-5 do artigo, com lr 1e-4, não encolhe nada) |
| Combinações | dropout + L2 | médio + 1e-4; pesado + 1e-3 |

O L1 e o L2 atuam nos mesmos pesos que recebem o weight decay. O tokenizador, o `[CLS]`, as normas e os vieses ficam de fora. A parada continua olhando só a entropia cruzada, sem a penalidade.

A **LayerNorm** já está em todo bloco do FT-Transformer. A **BatchNorm** fica de fora: ela normaliza pelo lote, não combina com os tokens de um Transformer e piorou o MLP em todas as camadas.

A régua é a mesma da fase 1, e o teste não entra. As linhas da fase 1, inclusive o trial 0, vêm do GitHub e não são medidas de novo.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/ft_transformer/ft_transformer_fase2.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"           # branch com o código (precisa estar publicada no GitHub)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "ft-fase2-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"   # vazio = procurar no repositório e em /kaggle/input
POR_GPU = 2                      # processos (dobras ao mesmo tempo) em cada GPU; sem GPU, processos na CPU
FASE1_RUN = "ft-fase1-20261112-2125"   # execução da fase 1 na branch de resultados (ou uma pasta local): traz o trial 0 já medido
EPOCAS_CURVA = 100              # épocas das curvas de sobreajuste (sem parada)
CURVAS = ("trial0", "r_drop_pesado", "r_l2_1e-3", "r_combo_pesado")   # configurações nas curvas
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa configurações novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "ft_transformer" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

PASTA_FT = REPO_DIR / "experimentos" / "ft_transformer"
sys.path[:0] = [str(REPO_DIR), str(PASTA_FT)]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(PASTA_FT), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("ft-fase2-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"          # antes de importar modelo.py, que lê o limite na importação
    EPOCAS_CURVA = 3
    print("MODO_TESTE: 3 épocas e duas regularizações. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import torch
import comum, modelo as M, fase1 as F1, fase2 as F2

DISPOSITIVOS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
if DISPOSITIVOS == ["cpu"]:
    POR_GPU = os.cpu_count() or 1                 # sem GPU: um processo por núcleo
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | dispositivos: {DISPOSITIVOS} x {POR_GPU} processos | até {M.MAX_EPOCAS} épocas")

# O envio ao GitHub é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="ft_transformer", config=dict(modelo="ft_transformer", fase=1, dispositivos=DISPOSITIVOS,
                                                               por_gpu=POR_GPU, teste=MODO_TESTE, padrao=M.PADRAO))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def ao_registrar(r):
    """Cada configuração medida vai para o W&B e para o GitHub na hora: se o Kaggle cair, nada se perde."""
    if wandb_run is not None:
        v = r.get("vs_trial0") or {}
        wandb_run.log(dict(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                           mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), epocas=r.get("epocas"),
                           minutos=r.get("tempo_min"), delta_vs_trial0=v.get("delta")))
    salvar()


# As linhas da fase 1 (trial 0, grade e sem oversampling) entram no placar desta execução sem treinar de novo
import tempfile
if FASE1_RUN and not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    with tempfile.TemporaryDirectory() as _tmp:
        ok = (shutil.copytree(FASE1_RUN, _tmp, dirs_exist_ok=True) and True) if os.path.isdir(FASE1_RUN) else \
            fluxo.baixar_github(_tmp, FASE1_RUN, RESULTS_REPO_URL or REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
        if ok and os.path.isfile(os.path.join(_tmp, "placar.jsonl")):
            shutil.copy(os.path.join(_tmp, "placar.jsonl"), PASTA)
            shutil.copytree(os.path.join(_tmp, "placar_oof"), os.path.join(PASTA, "placar_oof"), dirs_exist_ok=True)
fase1_linhas = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"}
print(f"Fase 1: {len(fase1_linhas)} linhas trazidas de {FASE1_RUN}" if fase1_linhas else "Fase 1: nada trazido; o trial 0 será medido aqui")


def medir(itens):
    return F1.garantir(PASTA, itens, DISPOSITIVOS, POR_GPU, continuar=continuar, ao_registrar=ao_registrar)


def mostrar(linhas):
    display(F1.tabela(linhas).round(4))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F1.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
Xtr_, _, ytr_, _ = F1._dados()
Xa_, _, _ = comum.preparar(Xtr_, ytr_, Xtr_.iloc[:5])
n_param = sum(p.numel() for p in M.FTTransformer(Xa_.shape[1], **M.arquitetura(M.PADRAO)).parameters())
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões sem sobreposição | assinatura do treino {F1.assinatura()}")
print(f"{Xa_.shape[1]} features = {Xa_.shape[1]} tokens + [CLS] | FT-Transformer padrão: {n_param:,} parâmetros para {len(Xtr_):,} clientes")

# 1. Dá para ver o sobreajuste?

Cada configuração treina na parte de ajuste da dobra 0 por um número fixo de épocas, **sem parada**. A cada época, a figura mostra a entropia cruzada em duas partes:

- **treino:** os clientes do ajuste, sem as cópias do oversampling;
- **parada:** os 20% internos que decidem a parada no placar.

A linha pontilhada marca a época de menor entropia na parada, que é onde a parada pararia. Sobreajuste aparece como uma tesoura: o treino continua caindo e a parada volta a subir. Se a regularização funcionar, a tesoura abre mais tarde ou menos.

São curvas de uma dobra só e servem para enxergar o mecanismo. Quem decide é o placar, na seção 2.

In [ ]:
CONFIGS = {"trial0": F1.TRIAL0, **F2.REGULARIZACAO}
arq_curvas = os.path.join(PASTA, "curvas_regularizacao.json")
cvs = json.load(open(arq_curvas)) if os.path.isfile(arq_curvas) else {}
t0 = time.time()
for r in CURVAS:
    if r not in cvs and continuar():
        cvs[r] = F2.curvas(CONFIGS[r], k=0, epocas=EPOCAS_CURVA, dispositivo=DISPOSITIVOS[0])
        json.dump(cvs, open(arq_curvas, "w"))
print(f"{(time.time() - t0) / 60:.1f} min")
fig = F2.figura_curvas({r: cvs[r] for r in CURVAS if r in cvs}, os.path.join(PASTA, "curvas_regularizacao.png"))
plt.show()
for r, v in cvs.items():
    i = v["melhor_epoca"] - 1
    print(f"{r:<18} menor CE na parada na época {v['melhor_epoca']:>3} ({v['ce_parada'][i]:.4f}, KS {v['ks_parada'][i]:.4f}) | "
          f"no fim: treino {v['ce_treino'][-1]:.4f}, parada {v['ce_parada'][-1]:.4f}")
_ = salvar()

# 2. Algum regularizador ganha do trial 0 no placar?

Cada configuração passa pela mesma régua da fase 1, com 15 dobras, parada nos 20% internos e veredito pareado contra o trial 0. Além do KS, vale olhar duas colunas:

- **`epocas`:** se a regularização deixou a rede treinar mais antes de parar;
- **`mse_treino` contra `mse_val`:** se a distância entre os dois diminuiu.

In [ ]:
T0 = ("ft_transformer", "trial0")
REG = F2.REGULARIZACAO if not MODO_TESTE else {k: F2.REGULARIZACAO[k] for k in ("r_drop_pesado", "r_l2_1e-3")}
t0 = time.time()
linhas_t0 = medir([("trial0", F1.TRIAL0, None)])
linhas_reg = medir([(r, c, {"vs_trial0": T0}) for r, c in REG.items()])
print(f"{(time.time() - t0) / 60:.1f} min")
tr = F1.tabela(linhas_t0 + linhas_reg)
reg = {r: c["hp"] for r, c in {"trial0": F1.TRIAL0, **REG}.items()}
tr.insert(1, "dropout", [f"{reg[r]['drop_atencao']}/{reg[r]['drop_ffn']}/{reg[r]['drop_residual']}" for r in tr.rotulo])
tr.insert(2, "l2", [reg[r].get("l2", 0) for r in tr.rotulo])
tr.insert(3, "l1", [reg[r].get("l1", 0) for r in tr.rotulo])
tr.insert(4, "wd", [reg[r]["wd"] for r in tr.rotulo])
display(tr.drop(columns=["blocos", "d_token", "over"]).round(4))

# 3. Placar das fases 1 e 2

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "ft_transformer"]
tab = F1.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_ft_transformer_fase2.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_trial0") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do trial 0: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 2

- **As curvas mostram a tesoura? A regularização atrasou ou reduziu o sobreajuste?** _…_
- **Algum regularizador ganhou do trial 0 no placar?** _…_
- **A regularização deixou a rede treinar por mais épocas? O KS acompanhou?** _…_
- **MSE no treino contra a validação:** _…_
- **Próxima hipótese:** _…_